In [ ]:
import pandas as pd

import sys
sys.path.append("../src")

from spatiotemporal_yield_loss_estimation import *

In [ ]:
vulnerability_params_df = pd.read_csv(
    "../outputs/3_vuln_curve_parameters/"
    "crop_country_vulnerability_curve_parameters.csv"
)

## run case per country
For rice/maize/sugarcane/cassava

In [ ]:
yield_fao_df = pd.read_csv(
    "../data/yield_fao-owid_adm0_processed/yield_cassava.csv"
)
crop_name = "cassava"

In [ ]:
country_code = "vn"

grid_df = pd.read_csv(
    f"../outputs/3.3_vuln_curve_samples/"
    f"vuln_curve_samples_cassava_vn_260720.csv"
)

grid_yield_output_path = (
    f"../outputs/4.1_spatiotemporal_yield_loss/"
    f"grid_yield_{crop_name}_{country_code}.csv"
)
country_yield_output_path = (
    f"../outputs/4.1_spatiotemporal_yield_loss/"
    f"country_yield_{crop_name}_{country_code}.csv"
)

In [ ]:
yield_df = yield_fao_df[["Year", country_code]].copy()
yield_df.columns = ["year", "yield_t_ha"]

yield_df["yield_trend_t_ha"] = get_trend_polynomial(yield_df["yield_t_ha"], degree=2)

yield_df["yield_detrended_t_ha"] = yield_df["yield_t_ha"] - yield_df["yield_trend_t_ha"]

yield_normal_detrended = yield_df["yield_detrended_t_ha"].mean()
yield_df["yield_normal_t_ha"] = (
    yield_df["yield_trend_t_ha"] + yield_normal_detrended
)
print(
    f"Mean detrended yield for {country_code} {crop_name}: "
    f"{yield_normal_detrended:.2f} t/ha"
)

In [ ]:
row = vulnerability_params_df[
    (vulnerability_params_df["crop_name"]==crop_name)
    & (vulnerability_params_df["country_code"]==country_code.upper())
    ].iloc[0]

B = row.B
k = row.k
x0 = row.x0

grid_df["ylr_vc"] = logistic_ranged(grid_df["spei"], B, k, x0)
grid_df["ylr_vc"] = grid_df["ylr_vc"].fillna(0)

print(f"Max YLR by vuln curve: {grid_df["ylr_vc"].max():.3f}")

grid_df["yield_vc_t_ha"] = (1-grid_df["ylr_vc"]) * yield_normal_detrended

In [ ]:
grid_yield_df = grid_df[["GID", "year", "harvest_area_ha", "yield_sim_t_ha"]].copy()

yearly_yield_ac_df = aggregate_yearly_yield(
    grid_yield_df,
    yield_threshold=0.01*yield_normal_detrended
)

yearly_yield_ac_df = yearly_yield_ac_df.rename(
    columns={"yield_sim_t_ha": "yield_sim_ac_t_ha"}
)

grid_yield_df = grid_df[["GID", "year", "harvest_area_ha", "yield_vc_t_ha"]].copy()
grid_yield_df = grid_yield_df.rename(
    columns={"yield_vc_t_ha": "yield_sim_t_ha"}
)

yearly_yield_vc_df = aggregate_yearly_yield(
    grid_yield_df,
    yield_threshold=0.01*yield_normal_detrended
)

yearly_yield_vc_df = yearly_yield_vc_df.rename(
    columns={"yield_sim_t_ha": "yield_sim_vc_t_ha"}
)

yield_df = yield_df.merge(
    yearly_yield_vc_df,
    on="year",
    how="right",
)

yield_df = yield_df.merge(
    yearly_yield_ac_df,
    on="year",
    how="left",
)

yield_df["yield_reconstructed_ac_t_ha"] = (
    yield_df["yield_sim_ac_t_ha"]+yield_df["yield_trend_t_ha"]
)
yield_df["yield_reconstructed_vc_t_ha"] = (
    yield_df["yield_sim_vc_t_ha"]+yield_df["yield_trend_t_ha"]
)
yield_df["ylr"] = 1 -  (
    yield_df["yield_reconstructed_vc_t_ha"]/yield_df["yield_normal_t_ha"]
)

In [ ]:
grid_df.to_csv(grid_yield_output_path, index=False)
yield_df.to_csv(country_yield_output_path, index=False)